In [1]:
import requests
import time
import pandas as pd
from pathlib import Path

# CVR API Base URL
BASE_URL = "https://cvrapi.dk/api"

# Default params for the API
# IMPORTANT: Provide a User-Agent identifying your application to avoid being blocked
headers = {
    "User-Agent": "DTU MSc HCAI - Project Social Data Analysis - Educational Use - Andrea De Pascale s243094@dtu.dk"
}

In [2]:
companies_data = pd.read_csv("../data/companies/full_list_companies.csv", sep=',', encoding='utf-8')
companies_list = companies_data['Company'].tolist()

In [3]:
#companies = companies_list[:36]
#companies = companies_list[36:86] # Uliyan's list
companies = companies_list[86:136] # Andreas' list
#companies = companies_list[136:] # max list 169 companies
#companies = companies_list[150:]

In [4]:
# List of companies to search for
results = []

for company in companies:
    print(f"Searching for: {company}")
    params = {
        "country": "dk",
        "search": company
    }
    
    response = requests.get(BASE_URL, params=params, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        if "error" in data:
            if data['error'] == 'QUOTA_EXCEEDED':
                print("  API quota exceeded. Stopping further requests.")
                break
            print(f"  Error for {company}: {data['error']}")
        else:
            results.append(data)
            print(f"  Success: Found {data.get('name', 'Unknown')}, CVR: {data.get('vat', 'Unknown')}")
    else:
        print(f"  Failed with status code: {response.status_code}")
        
    # Be nice to the API - add a short delay
    time.sleep(4)

print(f"\nFetched data for {len(results)} companies.")

Searching for: L'ORÉAL DANMARK
  Success: Found L'ORÉAL DANMARK A/S, CVR: 70710218
Searching for: LB FORSIKRING
  Success: Found LB FORSIKRING A/S, CVR: 16500836
Searching for: LEGO HOLDING
  Success: Found LEGO Holding A/S, CVR: 28122454
Searching for: LEGO SYSTEM
  Success: Found LEGO SYSTEM A/S, CVR: 47458714
Searching for: LEGO System
  Success: Found LEGO SYSTEM A/S, CVR: 47458714
Searching for: LEO FONDET
  Success: Found LEO FONDET, CVR: 11623336
Searching for: LEO Pharma
  Success: Found LEO PHARMA A/S, CVR: 56759514
Searching for: LOLLAND KOMMUNE
  Success: Found Lolland Kommune, CVR: 29188572
Searching for: LUNDBECKFONDEN
  Success: Found LUNDBECKFONDEN, CVR: 11814913
Searching for: Lagkagehuset
  Success: Found LAGKAGEHUSET A/S, CVR: 20213094
Searching for: Leverandørselskabet Danish Crown
  Success: Found LEVERANDØRSELSKABET DANISH CROWN AMBA, CVR: 21643939
Searching for: Lidl Danmark K/S
  Success: Found Lidl Danmark K/S, CVR: 26630797
Searching for: Lyngby-Taarbæk Kommune

In [5]:
data_rows = []
index_list = []
keys = ['name', 'address', 'zipcode', 'city', 'startdate', 'enddate', 'employees']
columns = ['CompanyVat', 'CompanyName', 'UnitName', 'UnitAddress', 'UnitZipcode', 'UnitCity', 'UnitStartdate', 'UnitEnddate', 'UnitEmployees']

for res in results:
    for unit in res.get('productionunits', []):
        pno = unit.get('pno', None)
        values = [res['vat'], res['name']] + [unit.get(key, None) for key in keys]
        
        data_rows.append(values)
        index_list.append(pno)

# Create DataFrame
df_units = pd.DataFrame(data_rows, columns=columns, index=index_list)
#df_units.set_index('pno', inplace=True)
df_units.index.name = 'pno'

#display(df_units.head())

In [6]:
# Save to CSV
output_dir = Path("../data/CVR_API")
output_dir.mkdir(parents=True, exist_ok=True)

# Check if output file exists, if so, appendt to the file, otherwise create a new one
output_file = output_dir / "Andrea_cvr_production_units.csv"
if output_file.exists():
    df_units.to_csv(output_file, mode='a', header=False, index=False, encoding='utf-8')
else:
    df_units.to_csv(output_file, index=False, encoding='utf-8')



In [7]:
# Save results from API to JSON
df_cvr = pd.DataFrame(results)

output_json_file = output_dir / "Andrea_cvr_production_units.json"
if output_json_file.exists():
    df_cvr.to_json(output_json_file, orient='records', indent=4, mode='a')
else:
    df_cvr.to_json(output_json_file, orient='records', indent=4)

In [8]:
# # Convert results to a pandas DataFrame and save to resources folder
# if results:
#     df_cvr = pd.DataFrame(results)
    
#     # Ensure resources directory exists
#     resources_dir = Path("../resources")
#     resources_dir.mkdir(parents=True, exist_ok=True)
    
#     output_path = resources_dir / "cvr_data.json"
    
#     # Save as JSON (preserves nested dictionary structures better than CSV)
#     df_cvr.to_json(output_path, orient="records", indent=4)
#     print(f"Saved corporate data to {output_path}")
    
#     # Also save a simplified CSV version for easier direct viewing
#     cols_to_save = ['vat', 'name', 'address', 'zipcode', 'city', 'phone', 'email', 'employees']
#     # Keep only available columns
#     available_cols = [col for col in cols_to_save if col in df_cvr.columns]
    
#     csv_out_path = resources_dir / "cvr_data.csv"
#     df_cvr[available_cols].to_csv(csv_out_path, index=False)
#     print(f"Saved simplified CSV to {csv_out_path}")
    
#     display(df_cvr[available_cols].head())
# else:
#     print("No data to save.")